# Hyperparameter Optimization: Random Search vs Grid Search with Nested Validation

**Dataset:** `digits` | **Level:** advanced

Does a larger search budget help, and how optimistic is the score of the best configuration?

Sections: 1 Problem, 2 Method, 3 Setup and data, 4 Experiment, 5 Results, 6 Interpretation, 7 Limitations, 8 Exercises, 9 References.

## 1. Problem

The best score found by a search is biased upward because it was selected as the maximum. Nested cross validation measures that bias.

## 2. Method

Support vector machine with C and gamma. Inner loop: random search with 5, 20 and 60 configurations versus a 6 by 6 grid. Outer loop: 5-fold stratified. We report the inner best score and the outer score on untouched folds.

## 3. Setup and data

Optical digits, 1797 rows, 64 features (see catalog).

In [1]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, KFold, cross_val_score
from ml_lab.data import load_dataset
from ml_lab.preprocessing import split_three_way, make_model_pipeline
from ml_lab.experiments import ExperimentRecord, save_figure
from ml_lab.utils import set_seed, print_environment
from ml_lab import visualization as viz
NB_ID = "hyperparameter_optimization"
SEED = 42
set_seed(SEED); viz.apply_style(); print_environment()

python 3.13.18 | Linux-6.18.44-fc-v595-x86_64-with-glibc2.39
  numpy==2.5.3
  pandas==3.0.5
  scipy==1.18.1
  scikit-learn==1.9.1
  matplotlib==3.11.2
  xgboost==3.4.1
  lightgbm==4.7.0
  catboost==1.2.10
  umap-learn==0.6.0
  torch==2.15.0
  gymnasium==1.4.0


In [2]:
b = load_dataset("digits"); X, y = b.X, b.y
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from scipy.stats import loguniform
pipe = make_pipeline(StandardScaler(), SVC())
print(X.shape)

(1797, 64)


## 4. Experiment

Run the outer loop for each search strategy.

In [3]:
outer = StratifiedKFold(5, shuffle=True, random_state=SEED)
strategies = {"random 5": lambda: RandomizedSearchCV(pipe, {"svc__C": loguniform(1e-2, 1e3), "svc__gamma": loguniform(1e-5, 1e-1)}, n_iter=5, cv=3, random_state=SEED, n_jobs=2),
              "random 20": lambda: RandomizedSearchCV(pipe, {"svc__C": loguniform(1e-2, 1e3), "svc__gamma": loguniform(1e-5, 1e-1)}, n_iter=20, cv=3, random_state=SEED, n_jobs=2),
              "random 60": lambda: RandomizedSearchCV(pipe, {"svc__C": loguniform(1e-2, 1e3), "svc__gamma": loguniform(1e-5, 1e-1)}, n_iter=60, cv=3, random_state=SEED, n_jobs=2),
              "grid 36": lambda: GridSearchCV(pipe, {"svc__C": np.logspace(-2, 3, 6), "svc__gamma": np.logspace(-5, -1, 6)}, cv=3, n_jobs=2)}
res_rows = []
for name, make in strategies.items():
    inner, outer_s = [], []
    for tr, te in outer.split(X, y):
        s = make().fit(X.iloc[tr], y.iloc[tr]); inner.append(s.best_score_); outer_s.append(s.score(X.iloc[te], y.iloc[te]))
    res_rows.append({"strategy": name, "inner_best_score": np.mean(inner), "outer_score": np.mean(outer_s), "outer_std": np.std(outer_s, ddof=1), "optimism": np.mean(inner) - np.mean(outer_s)})
print("done")

done


## 5. Results

Inner best versus outer score.

In [4]:
tab = pd.DataFrame(res_rows).set_index("strategy"); display(tab.round(4))
default = cross_val_score(pipe, X, y, cv=outer).mean()
print("default SVC (no search) outer accuracy:", round(float(default), 4))
METRICS = {f"{n.replace(' ', '_')}_{k}": float(v) for n, r in tab.iterrows() for k, v in r.items()}
METRICS["default_svc_accuracy"] = float(default)
CONFIG = {"outer_cv": 5, "inner_cv": 3, "model": "SVC RBF"}; EXTRA = {"table": tab.round(4).reset_index().to_dict("records")}; ALGORITHM = "random search vs grid search"

record = ExperimentRecord(notebook=NB_ID, family="advanced", algorithm=ALGORITHM, dataset="digits", task="classification", seed=SEED, config=CONFIG, metrics=METRICS, extra=EXTRA)
print("saved:", record.save())

,inner_best_score,outer_score,outer_std,optimism
strategy,,,,
random 5,0.9521,0.9828,0.0069,-0.0306
random 20,0.9542,0.9816,0.0072,-0.0274
random 60,0.9548,0.9822,0.0070,-0.0274
grid 36,0.9524,0.9844,0.0080,-0.0320


default SVC (no search) outer accuracy: 0.9839
saved: experiments/results/hyperparameter_optimization.json


## 6. Interpretation

The inner score is lower than the outer score here (negative optimism). The reason is that the inner folds train on 3-fold subsets of 80 percent of the data, so they see less data than the outer fit. The usual upward selection bias exists but is smaller than this effect, so the gap is not a clean estimate of optimism. All four strategies and the default SVC land within one outer standard deviation of each other, so on this dataset the search brings no measurable gain. That is a result about this easy dataset, not about search in general.

## 7. Limitations

One model and one dataset. Digits is an easy task, so the search has little room. Search costs are counted in configurations, not in measured seconds.

## 8. Exercises

1. Add successive halving (`HalvingRandomSearchCV`).
2. Repeat with five outer seeds and average.
3. Try the same on a harder dataset such as Telco.

## 9. References

Bergstra and Bengio (2012), *Random search for hyper-parameter optimization*, JMLR, motivates random search. It is **not verified in the build environment**.